# VoiceStudio on free Colab - Pro-style remote GPU

Your PC runs ONLY the UI. Every bit of CPU/GPU work happens here on the T4.
This is what "Pro" sells (remote GPU + all features + multiple devices), free.

## First time
1. Runtime -> Change runtime type -> **T4 GPU**
2. Runtime -> **Run all** (or run boxes 1, 2, 3 by hand)
3. Scroll to the END: the **URL** and **API key** are printed there.
4. On the PC: double-click the **VoiceStudio** icon, paste the URL, then
   paste the key (the launcher puts it on your clipboard).

Box 4 reprints the pair. Box 5 adds dubbing (needs your Hugging Face token).

## When Colab restarts
* Tab reconnect / kernel restart -> run **box 2** only, paste the new URL.
  (Same key: it lives on Google Drive, so your PC stays paired.)
* Fresh VM -> run **box 1** then **box 2**. Models are already on Drive.

## What is stored where
| What | Where |
|---|---|
| Models, settings, cloned voices | Google Drive (`/content/drive/omnivoice`) |
| API key | Google Drive - survives VM resets |
| Public URL | changes every fresh tunnel (that is the security) |
| Compute | Colab T4 only. Your PC does nothing. |

In [4]:
import io
import json
import os
import pathlib
import re
import subprocess
import tarfile
import time
import urllib.request

from google.colab import userdata

management_url = userdata.get("NETBIRD_MANAGEMENT_URL")
setup_key = userdata.get("NETBIRD_SETUP_KEY")

if not management_url or not setup_key:
    raise ValueError(
        "Crie os Secrets NETBIRD_MANAGEMENT_URL e NETBIRD_SETUP_KEY no Colab."
    )

# Baixa o binário Linux x86_64 da última release oficial.
request = urllib.request.Request(
    "https://api.github.com/repos/netbirdio/netbird/releases/latest",
    headers={"Accept": "application/vnd.github+json"},
)
with urllib.request.urlopen(request, timeout=30) as response:
    release = json.load(response)

asset = next(
    item for item in release["assets"]
    if re.fullmatch(r"netbird_.*_linux_amd64\.tar\.gz", item["name"])
)

with urllib.request.urlopen(asset["browser_download_url"], timeout=120) as response:
    archive_bytes = response.read()

with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as archive:
    binary_member = next(
        item for item in archive.getmembers()
        if pathlib.PurePosixPath(item.name).name == "netbird" and item.isfile()
    )
    binary_path = pathlib.Path("/usr/local/bin/netbird")
    binary_path.write_bytes(archive.extractfile(binary_member).read())
    binary_path.chmod(0o755)

# O modo foreground mantém o agente ativo durante a sessão do Colab.
env = os.environ.copy()
env.update({
    "NB_SETUP_KEY": setup_key,
    "NB_MANAGEMENT_URL": management_url,
    "NB_USE_NETSTACK_MODE": "true",
    "NB_ENABLE_NETSTACK_LOCAL_FORWARDING": "true",
})

log_file = open("/content/netbird.log", "ab", buffering=0)
netbird_process = subprocess.Popen(
    [str(binary_path), "up", "--foreground-mode", "--log-file", "console"],
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

time.sleep(3)
if netbird_process.poll() is not None:
    print(pathlib.Path("/content/netbird.log").read_text(errors="replace")[-4000:])
    raise RuntimeError("O NetBird encerrou durante a inicialização.")

print(f"NetBird iniciado. Versão: {release['tag_name']}")
print("Confira o estado com: !netbird status")
print("Logs: /content/netbird.log")


NetBird iniciado. Versão: v0.83.2
Confira o estado com: !netbird status
Logs: /content/netbird.log


In [2]:
# 1. Install the app on Colab (run once per VM; ~5-10 min)
INSTALL_GPU_TURBO = True   # ~2x faster generations. Set False to skip the extra download.
PINNED_COMMIT = "f47221a1f0af89df27c34747471daa30391a9487"  # tested with this notebook
# Keep models, settings and your API KEY on Google Drive, so when Colab recycles
# the VM (it will) everything is still there: no 5 GB re-download, and the key
# already saved on your PC keeps working - only the URL changes.
PERSIST_TO_GOOGLE_DRIVE = True

import importlib.metadata as _md, importlib.util, json, os, shutil, subprocess, sys, time

print("VoiceStudio setup: box 1/6 - install the app on this VM.")
print("Takes 5-10 minutes the first time. Do not close this tab.")

def run(cmd, cwd=None):
    print(f"\n$ {cmd if isinstance(cmd, str) else ' '.join(cmd)}", flush=True)
    rc = subprocess.run(cmd, cwd=cwd, shell=isinstance(cmd, str)).returncode
    if rc != 0:
        raise SystemExit(f"FAILED ({rc}). Re-run this box.")

# --- GPU check -------------------------------------------------------------
if not shutil.which("nvidia-smi"):
    raise SystemExit("No GPU found. Runtime -> Change runtime type -> T4 GPU, then re-run.")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv"], capture_output=True, text=True).stdout.strip())

REPO_DIR = "/content/VoiceStudio"
REPO_URL = "https://github.com/debpalash/VoiceStudio"
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")

# --- Clone (pinned so this notebook always matches the code it targets) -----
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    for attempt in (1, 2, 3):        # Colab->GitHub drops connections often
        rc = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                            capture_output=True, text=True).returncode
        if rc == 0:
            break
        print(f"git clone attempt {attempt}/3 failed, retrying in 5s...")
        subprocess.run("rm -rf " + REPO_DIR, shell=True)
        time.sleep(5)
    else:
        raise SystemExit("Could not download the app from GitHub (network). Re-run this box.")
_head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                       capture_output=True, text=True).stdout.strip()
if _head != PINNED_COMMIT and not os.environ.get("COLAB_UNPINNED"):
    run(["git", "fetch", "--depth", "1", "origin", PINNED_COMMIT], cwd=REPO_DIR)
    run(["git", "checkout", "-q", PINNED_COMMIT], cwd=REPO_DIR)
    _head = PINNED_COMMIT
print("Repo at", _head[:12])

# --- ffmpeg (dubbing/decoding) --------------------------------------------
if shutil.which("ffmpeg"):
    print("ffmpeg present.")
else:
    for attempt in range(1, 4):
        if subprocess.run("apt-get update -qq", shell=True).returncode == 0:
            break
        print(f"apt-get update retry {attempt}/3 in 5s...")
        time.sleep(5)
    run("apt-get install -y -qq --fix-missing ffmpeg libsndfile1")

# --- App + engine dependencies --------------------------------------------
if not shutil.which("uv"):
    run([sys.executable, "-m", "pip", "install", "-q", "uv"])
for attempt in (1, 2):              # a 10-minute step must not die on one hiccup
    rc = subprocess.run(["uv", "pip", "install", "--system", "--no-cache",
                         "--constraint", "deploy/torch-constraints.txt", "."],
                        cwd=REPO_DIR).returncode
    if rc == 0:
        break
    print(f"dependency install attempt {attempt}/2 failed - retrying...")
    time.sleep(10)
else:
    raise SystemExit("Dependency install failed twice. Re-run this box.")
os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)
run([sys.executable, os.path.join(REPO_DIR, "scripts", "setup.py")])
run([sys.executable, "-c",
     "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"],
    cwd=REPO_DIR)

# --- GPU Turbo (optional; never allowed to break the install) -------------
if INSTALL_GPU_TURBO and importlib.util.find_spec("flashinfer") is None:
    try:
        import torch as _torch_probe
        _cu = (_torch_probe.version.cuda or "12.8").replace(".", "")   # "12.8" -> "128"
        _idx = f"https://flashinfer.ai/whl/cu{_cu}"
        print(f"Installing GPU Turbo (FlashInfer) from {_idx} ...")
        _pin = "/content/turbo-constraints.txt"
        with open(_pin, "w") as _f:
            _f.write(f"torch=={_md.version('torch')}\n")
            _f.write(f"torchaudio=={_md.version('torchaudio')}\n")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                        "-c", _pin, "flashinfer-python", "flashinfer-jit-cache",
                        "--extra-index-url", _idx], check=False)
        # A broken install must never slow generations: import-test it, and
        # remove the package outright if it cannot even load.
        if importlib.util.find_spec("flashinfer") is not None:
            subprocess.run([sys.executable, "-c", "import flashinfer"], check=True)
        else:
            raise RuntimeError("no flashinfer module after install")
    except Exception as exc:  # noqa: BLE001 - optional feature, never fatal
        print("GPU Turbo not available (%s). Continuing without it." % str(exc)[:160])
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "flashinfer-python", "flashinfer-jit-cache"], check=False)
if PERSIST_TO_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as exc:  # noqa: BLE001 - install is fine without Drive
        print("Drive not mounted (%s)." % str(exc)[:120])
        print("Continuing WITHOUT persistence: a Colab reset will mean a new key")
        print("and re-downloaded models.")
        PERSIST_TO_GOOGLE_DRIVE = False

if PERSIST_TO_GOOGLE_DRIVE:
    # Modified to save inside MyDrive as creating files directly in /content/drive root is not supported by Google Drive FUSE.
    root = "/content/drive/MyDrive/omnivoice"
    os.makedirs(root + "/hf", exist_ok=True)
    os.makedirs(root + "/data", exist_ok=True)    # settings, licences, cloned voices

    # The HF cache normally stores weights as symlinks into blobs/, which
    # Drive's FUSE mount refuses. HF_HUB_DISABLE_SYMLINKS=1 makes huggingface_hub
    # COPY the files instead, while keeping the same snapshots/ layout that the
    # app's is_cached() detector understands - so the cache CAN live on Drive.
    # The flag is read once at import time, so it is set before anything imports
    # huggingface_hub.
    os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"
    hf_dir = root + "/hf"
    os.makedirs(hf_dir, exist_ok=True)
    with open(root + "/config.json", "w") as fh:
        json.dump({"hf": hf_dir, "data": root + "/data", "root": root,
                   "models_on_drive": hf_dir == root + "/hf"}, fh, indent=1)
    os.environ["HF_HOME"] = hf_dir
    print("Drive persistence ON at", root)
    print("  settings/voices on Drive:", root + "/data")
    print("  model cache            :", hf_dir,
          "(copy mode: Drive FUSE cannot hold symlinks, so HF copies instead)")

print("Box 1 done.")

VoiceStudio setup: box 1/6 - install the app on this VM.
Takes 5-10 minutes the first time. Do not close this tab.
name, memory.total [MiB]
Tesla T4, 15360 MiB
Repo at f47221a1f0af
ffmpeg present.

$ /usr/bin/python3 /content/VoiceStudio/scripts/setup.py

$ /usr/bin/python3 -c import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive persistence ON at /content/drive/MyDrive/omnivoice
  settings/voices on Drive: /content/drive/MyDrive/omnivoice/data
  model cache            : /content/drive/MyDrive/omnivoice/hf (copy mode: Drive FUSE cannot hold symlinks, so HF copies instead)
Box 1 done.


In [15]:
# 2. Start the backend on NetBird VPN (No Cloudflare public tunnel)
PORT = 3900
LOG = "/content/omnivoice_backend.log"
KEY_FILE = "/content/omnivoice_api_key.txt"
URL_FILE = "/content/last_tunnel_url.txt"

import importlib.util, json, os, re, secrets, socket, subprocess, sys, threading, time, urllib.error, urllib.request

REPO_DIR = "/content/VoiceStudio"
DRIVE_ROOT = "/content/drive/MyDrive/omnivoice"
DRIVE = {}

# --- Mount Drive for persistence ---
if not os.path.isdir("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        print("Drive remounted for this session.")
    except Exception:
        pass
try:
    if os.path.exists(os.path.join(DRIVE_ROOT, "config.json")):
        DRIVE = json.load(open(os.path.join(DRIVE_ROOT, "config.json")))
        os.environ["HF_HOME"] = DRIVE["hf"]
        print("Drive persistence active:", DRIVE["hf"])
except Exception as exc:
    print("Drive state unreadable (%s); running without it." % str(exc)[:120])
    DRIVE = {}

if not os.path.isdir(os.path.join(REPO_DIR, "backend")):
    raise SystemExit("The app is not installed in this VM yet. Run box 1 first.")

def get_netbird_ip():
    """Obtém o IP do NetBird atribuído a esta máquina Colab."""
    try:
        output = subprocess.check_output(["netbird", "status"], text=True)
        match = re.search(r"IP:\s+([0-9]+\.[0-9]+\.[0-9]+\.[0-9]+)", output)
        if match:
            return match.group(1)
    except Exception:
        pass
    # Fallback obtendo diretamente da interface de rede se o cli falhar
    try:
        import socket
        return socket.gethostbyname(socket.gethostname())
    except Exception:
        return "0.0.0.0"

def _free_port(port):
    import signal
    killed = []
    for entry in os.listdir("/proc"):
        if not entry.isdigit():
            continue
        try:
            with open("/proc/%s/cmdline" % entry, "rb") as fh:
                argv = fh.read().decode("utf8", "replace").split("\0")
        except OSError:
            continue
        try:
            ours = ("uvicorn" in argv and "--port" in argv
                    and argv[argv.index("--port") + 1] == str(port))
        except (ValueError, IndexError):
            ours = False
        if ours:
            try:
                os.kill(int(entry), signal.SIGKILL)
                killed.append(entry)
            except OSError:
                pass
    subprocess.run("fuser -k %s/tcp 2>/dev/null || true" % port, shell=True)
    if killed:
        print("Freed port %s from stale backend(s): %s" % (port, ", ".join(killed)))

def start_backend(key):
    env = os.environ.copy()
    env.pop("OMNIVOICE_API_KEY", None)
    env["OMNIVOICE_API_KEY"] = key
    env["OMNIVOICE_SERVER_MODE"] = "1"
    env["OMNIVOICE_DATA_DIR"] = DRIVE.get("data", "/content/omnivoice_data")
    if "hf" in DRIVE:
        env["HF_HOME"] = DRIVE["hf"]
    # Permite acesso via IP do NetBird ou wildcard já que a VPN protege o acesso
    env["OMNIVOICE_ALLOWED_HOSTS"] = "*"
    env["PYTHONUNBUFFERED"] = "1"
    if DRIVE:
        env["HF_HUB_DISABLE_SYMLINKS"] = "1"
    if importlib.util.find_spec("flashinfer") is not None:
        env["OMNIVOICE_FLASHINFER"] = "graph"
        print("GPU Turbo: FlashInfer active.")
    else:
        env.pop("OMNIVOICE_FLASHINFER", None)
    _free_port(PORT)
    time.sleep(2)
    open(LOG, "w").close()
    log = open(LOG, "ab")
    # Escuta em 0.0.0.0 para aceitar conexões externas da rede virtual (NetBird)
    return subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app", "--app-dir", "backend",
         "--host", "0.0.0.0", "--port", str(PORT)],
        cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT)

def wait_health(url, seconds=300):
    deadline = time.time() + seconds
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url + "/health", timeout=5) as r:
                return json.load(r)
        except Exception:
            time.sleep(3)
    raise SystemExit("Backend did not come up. Run: !tail -40 /content/omnivoice_backend.log")

# --- The API key ---
KEY = ""
for _path in ([os.path.join(DRIVE_ROOT, "api_key.txt")] if DRIVE else []) + [KEY_FILE]:
    try:
        _candidate = open(_path).read().strip()
    except OSError:
        continue
    if len(_candidate) >= 32:
        KEY = _candidate
        break
if len(KEY) < 32:
    KEY = secrets.token_urlsafe(32)
open(KEY_FILE, "w").write(KEY)
if DRIVE:
    with open(os.path.join(DRIVE_ROOT, "api_key.txt"), "w") as _fh:
        _fh.write(KEY)
    print("API key stored on Drive.")

# Inicia o backend ouvindo na rede local/VPN
proc = start_backend(KEY)
netbird_ip = get_netbird_ip()
local_url = f"http://{netbird_ip}:{PORT}"

print(f"Aguardando inicialização no IP Netbird: {local_url}...")
info = wait_health(local_url)
print("Backend ready:", info)

# --- Keep Colab awake ---
def _heartbeat():
    while True:
        time.sleep(120)
        print("[alive]", time.strftime("%H:%M:%S"), "- keep this tab open", flush=True)
threading.Thread(target=_heartbeat, daemon=True).start()

print("=" * 66)
print("COPY BOTH - from this same run, below:")
print("=" * 66)
print("URL (NetBird VPN IP):\n" + local_url)
print("\nAPI KEY:\n" + KEY)
print("\nNo seu computador: certifique-se de estar conectado ao mesmo NetBird VPN.")
print("Abra o app do VoiceStudio no PC e cole o IP acima + a Chave (KEY).")
print("=" * 66)

Drive persistence active: /content/drive/MyDrive/omnivoice/hf
API key stored on Drive.
GPU Turbo: FlashInfer active.
Aguardando inicialização no IP Netbird: http://172.28.0.12:3900...
Backend ready: {'status': 'ok', 'device': 'cuda (Tesla T4)', 'version': '0.5.7'}
COPY BOTH - from this same run, below:
URL (NetBird VPN IP):
http://172.28.0.12:3900

API KEY:
w8hAobM0wSc4bgU6yxJajP9uB8xbkdO-7LzGx_B4JEQ

No seu computador: certifique-se de estar conectado ao mesmo NetBird VPN.
Abra o app do VoiceStudio no PC e cole o IP acima + a Chave (KEY).


In [16]:
# 3. One-time app setup: models, licences, privacy, Max quality (run after box 2)
#    Safe to re-run: downloads are cached and every step is idempotent.

import json, os, re, subprocess, urllib.error, urllib.request

def get_working_base():
    # Internamente, a máquina do Colab acessa melhor via localhost (127.0.0.1),
    # mesmo com o NetBird ativo em modo Netstack.
    for url in ["http://127.0.0.1:3900", "http://localhost:3900", "http://172.28.0.12:3900"]:
        try:
            req = urllib.request.Request(f"{url}/health", method="GET")
            with urllib.request.urlopen(req, timeout=3) as r:
                return url
        except Exception:
            continue

    try:
        output = subprocess.check_output(["netbird", "status"], text=True)
        match = re.search(r"IP:\s+([0-9]+\.[0-9]+\.[0-9]+\.[0-9]+)", output)
        if match:
            return f"http://{match.group(1)}:3900"
    except Exception:
        pass
    return "http://127.0.0.1:3900"

BASE = get_working_base()
DRIVE_ROOT = "/content/drive/MyDrive/omnivoice"

def call(path, body=None, method=None):
    req = urllib.request.Request(
        BASE + path,
        data=None if body is None else json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method=method or ("PUT" if body is not None else "GET"))
    with urllib.request.urlopen(req, timeout=90) as r:
        raw = r.read().decode()
    return json.loads(raw) if raw.strip() else {}

try:
    print(f"Tentando se conectar ao backend em: {BASE}...")
    info = call("/health")
    print("Conexão bem-sucedida! Backend info:", info)
except Exception as exc:
    raise SystemExit(f"Erro ao conectar no backend em {BASE} ({exc}). Certifique-se de que a Célula 2 está rodando e ativa!")

# --- Re-apply the persistence config from box 1 ---------------------------
CACHE = os.environ.get("HF_HOME", os.path.expanduser("~/.cache/huggingface"))
if not os.path.isdir("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass
try:
    cfg = json.load(open(os.path.join(DRIVE_ROOT, "config.json")))
    if cfg.get("hf"):
        CACHE = cfg["hf"]
except Exception:
    pass
os.environ["HF_HOME"] = CACHE
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS", "1")   # read at import time
print("model cache:", CACHE)

# --- 1) Models -------------------------------------------------------------
from huggingface_hub import snapshot_download
MODELS = [
    "k2-fsa/OmniVoice",                                  # voice + TTS
    "eustlb/higgs-audio-v2-tokenizer",                   # TTS tokenizer
    "Systran/faster-whisper-large-v3",                   # transcription
    "csukuangfj/sherpa-onnx-whisper-tiny",               # dictation
    "csukuangfj/sherpa-onnx-nemo-parakeet-tdt-0.6b-v3-int8",
    "facebook/nllb-200-distilled-600M",                  # translation
]
failed = []
for repo in MODELS:
    try:
        snapshot_download(repo)
        print(f"  ready: {repo}")
    except Exception as exc:  # keep going
        failed.append(repo)
        print(f"  FAILED: {repo} - {type(exc).__name__}: {str(exc)[:120]}")
if failed:
    print("")
    print("Some downloads failed (network?). Re-run this box - finished ones are cached.")

# --- 2) Licence acceptance (YOUR decision) --------------------------------
ACCEPT_MODEL_LICENCES = True
if ACCEPT_MODEL_LICENCES:
    for repo in ["k2-fsa/OmniVoice", "eustlb/higgs-audio-v2-tokenizer"]:
        details = call(f"/models/licenses/details/{repo}")
        fp = (details.get("acceptance") or {}).get("fingerprint")
        if not fp:
            print(f"  {repo}: no fingerprint published - skipping")
            continue
        call("/models/licenses/accept",
             {"repo_id": repo, "fingerprint": fp, "accepted": True},
             method="POST")
        print(f"  licence accepted: {repo}")
else:
    print("licences: NOT accepted yet.")
    print("  Read them in the app under Model Catalogue, then set")
    print("  ACCEPT_MODEL_LICENCES = True above and re-run this box.")

# --- 3) Privacy choice -----------------------------------------------------
SHARE_ANONYMOUS_STATS = False   # False = nothing is ever sent
state = call("/api/settings/analytics", {"enabled": bool(SHARE_ANONYMOUS_STATS)}, "PUT")
print("  analytics:", json.dumps(state))

# --- 4) Max quality profile ------------------------------------------------
try:
    profile = call("/api/settings/performance-profile", {"tier": "max"})
except urllib.error.HTTPError as exc:
    if exc.code == 409:
        raise SystemExit("A job is still running - finish it, then re-run this box.")
    raise
tier = profile.get("global")
tts = (profile.get("targets") or {}).get("tts") or {}
print("  tier:", tier, "| TTS:", json.dumps(tts))

problems = list(failed)
if tier != "max":
    problems.append("the Max tier did not apply (backend returned %r)" % tier)
elif tts.get("steps") != 64:
    problems.append("Max applied but TTS steps are %r, expected 64" % tts.get("steps"))

print("")
if problems:
    print("=" * 66)
    print("SETUP INCOMPLETE - fix these, then re-run this box:")
    for item in problems:
        print("  -", item)
    print("=" * 66)
    raise SystemExit(1)

print("=" * 66)
print("SETUP FINISHED.")
print("On the PC: double-click the VoiceStudio icon,")
print("paste the URL from box 2, then paste the API key (Ctrl+V).")
print("=" * 66)

Tentando se conectar ao backend em: http://127.0.0.1:3900...
Conexão bem-sucedida! Backend info: {'status': 'ok', 'device': 'cuda (Tesla T4)', 'version': '0.5.7'}
model cache: /content/drive/MyDrive/omnivoice/hf


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

  ready: k2-fsa/OmniVoice


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

... (9 outputs left out)

  ready: csukuangfj/sherpa-onnx-nemo-parakeet-tdt-0.6b-v3-int8


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

  ready: facebook/nllb-200-distilled-600M
  licence accepted: k2-fsa/OmniVoice
  licence accepted: eustlb/higgs-audio-v2-tokenizer
  analytics: {"enabled": false, "opted_in": false, "available": true, "prompted": true}
[alive] 01:48:47 - keep this tab open
  tier: max | TTS: {"steps": 64, "postprocess": true, "model_policy": "tts-first-installed"}

SETUP FINISHED.
On the PC: double-click the VoiceStudio icon,
paste the URL from box 2, then paste the API key (Ctrl+V).


In [ ]:
# 4. Reprint your URL + key (any time you lose them; app must still be running)
#    Free to run any time. Does NOT restart anything.
# Prefer the Drive copy of the key: that is the one that outlived the VM and
# therefore the one already saved on your PC. Fall back to this VM's copy.
def _read(*paths):
    for path in paths:
        try:
            value = open(path).read().strip()
        except OSError:
            continue
        if value:
            return value
    return ""

key = _read("/content/drive/MyDrive/omnivoice/api_key.txt", "/content/omnivoice_api_key.txt")
url = _read("/content/last_tunnel_url.txt")
if not url or not key:
    print("No saved pair here - run box 2 first.")
else:
    print("URL:\n" + url)
    print("\nAPI KEY:\n" + key)
    print("\nDouble-click the VoiceStudio icon on your PC and paste both.")

In [ ]:
# 5. OPTIONAL: dubbing pack - speaker separation (pyannote)
#    Needed only for Dubbing (and multi-speaker transcripts).
#
#    This one needs YOU, once, on the web:
#      1. Create a free account at https://huggingface.co and log in.
#      2. Open https://huggingface.co/pyannote/speaker-diarization-3.1
#         and click the green "Agree and access repository" button.
#      3. Open https://huggingface.co/pyannote/segmentation-3.0
#         and click the same button.
#      4. Go to https://huggingface.co/settings/tokens ->
#         "Create new token" -> pick the READ role -> copy it.
#    Paste the token below (it stays on this VM, it is never sent anywhere else).

HF_TOKEN = ""

import importlib.util, json, os, re, subprocess, sys, urllib.request

BASE = "http://127.0.0.1:3900"
PYANNOTE_REPOS = ["pyannote/speaker-diarization-3.1", "pyannote/segmentation-3.0",
                  "pyannote/wespeaker-voxceleb-resnet34-LM"]

def call(path, body=None, method=None):
    req = urllib.request.Request(
        BASE + path,
        data=None if body is None else json.dumps(body).encode(),
        headers={"Content-Type": "application/json"},
        method=method or ("PUT" if body is not None else "GET"))
    with urllib.request.urlopen(req, timeout=90) as r:
        raw = r.read().decode()
    return json.loads(raw) if raw.strip() else {}

# Point downloads at exactly the cache the backend reads. Without this, a
# kernel restart silently sends them to ~/.cache while the backend looks in
# Drive, and dubbing then reports the models as missing.
DRIVE_ROOT = "/content/drive/MyDrive/omnivoice"
if not os.path.isdir("/content/drive"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        pass
try:
    _cfg = json.load(open(os.path.join(DRIVE_ROOT, "config.json")))
    if _cfg.get("hf"):
        os.environ["HF_HOME"] = _cfg["hf"]
except Exception:
    pass
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS", "1")
print("model cache:", os.environ.get("HF_HOME", "default"))

token = (HF_TOKEN or "").strip()
if not token.startswith("hf_"):
    raise SystemExit("Paste your Hugging Face READ token above (hf_...) and re-run this box.")
os.environ["HF_TOKEN"] = token
os.environ["HUGGINGFACE_HUB_TOKEN"] = token

print("Saving the token to the backend ...")
print("  ", json.dumps(call("/api/settings/hf-token", {"token": token}, "POST"))[:200])

from huggingface_hub import snapshot_download
for repo in PYANNOTE_REPOS:
    snapshot_download(repo)          # small: config + embedding model
    print(f"  ready: {repo}")

for repo in PYANNOTE_REPOS:          # licence acceptance, same as box 3
    details = json.dumps(call(f"/models/licenses/details/{repo}"))
    match = re.search(r"v2:[a-f0-9]{64}", details)
    if match:
        call("/models/licenses/accept",
             {"repo_id": repo, "fingerprint": match.group(0), "accepted": True},
             method="POST")
        print(f"  licence accepted: {repo}")

if importlib.util.find_spec("pyannote") is None:   # pinned so torch never changes
    # Pin to the EXACT installed versions. A loose floor like torch>=2.8 would
    # let pip pull a different torch build over the one box 1 installed, and the
    # compiled CUDA stack would stop matching it.
    import importlib.metadata as _md
    pin = "/content/diarization-constraints.txt"
    with open(pin, "w") as fh:
        for _pkg in ("torch", "torchaudio"):
            try:
                fh.write(f"{_pkg}=={_md.version(_pkg)}\n")
            except _md.PackageNotFoundError:
                pass
    print("  pinned:", open(pin).read().split())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input",
                    "-c", pin, "pyannote.audio"], check=False)
if importlib.util.find_spec("pyannote") is None:
    raise SystemExit("pyannote.audio could not be installed - share the error text.")

profile = call("/api/settings/performance-profile", {"tier": "max"})
print("  tier:", profile.get("global"),
      "| diarisation:", profile.get("selections", {}).get("diarisation", {}).get("engine"))
print("Box 4 done - Dubbing is ready (first run loads the model: ~1 min).")